# EW Smart Scan V2 — 20-band contextual bandit

End-to-end model workflow: restricted data download, whole-file leakage-safe splits, 500 μs preprocessing, three LinUCB candidates, validation-only selection, round-robin comparison, inference logs, and an optional sealed final test. DQN is not part of V2.

In [ ]:
#@title Experiment controls
TRAIN_FILE_COUNT = 30 #@param {type:"integer"}
VALIDATION_FILE_COUNT = 10 #@param {type:"integer"}
TEST_FILE_COUNT = 10 #@param {type:"integer"}
TRAIN_START_ID = 0 #@param {type:"integer"}
VALIDATION_START_ID = 0 #@param {type:"integer"}
TEST_START_ID = 27 #@param {type:"integer"}
RUN_FINAL_TEST = False #@param {type:"boolean"}
PARALLEL_CANDIDATES = 3 #@param {type:"integer"}
assert min(TRAIN_FILE_COUNT, VALIDATION_FILE_COUNT, TEST_FILE_COUNT) > 0
assert 1 <= PARALLEL_CANDIDATES <= 3

## 1. Upload and install the project archive

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith('.zip')), None)
if archive is None:
    raise ValueError('Upload smart_scan_colab_project.zip')
workspace = Path('/content/smart_scan_v2')
if workspace.exists(): shutil.rmtree(workspace)
workspace.mkdir()
with zipfile.ZipFile(archive) as bundle: bundle.extractall(workspace)
project = next((p for p in [workspace, *workspace.iterdir()] if (p/'pyproject.toml').is_file()), None)
if project is None: raise FileNotFoundError('pyproject.toml missing from archive')
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Project root:', Path.cwd())

## 2. Create whole-file manifests (no pulse/window splitting)

In [ ]:
import json
DATASET_ID = 'alan-turing-institute/turing-synthetic-radar-dataset'
manifest_dir = Path('data/manifests'); manifest_dir.mkdir(parents=True, exist_ok=True)
def records(split, start, count):
    return [{'split': split, 'mode': 'stare', 'config_id': i} for i in range(start, start+count)]
development = {'dataset_id': DATASET_ID, 'description': 'V2 development only',
               'files': records('train', TRAIN_START_ID, TRAIN_FILE_COUNT) + records('val', VALIDATION_START_ID, VALIDATION_FILE_COUNT)}
sealed_test = {'dataset_id': DATASET_ID, 'description': 'V2 sealed test',
               'files': records('test', TEST_START_ID, TEST_FILE_COUNT)}
dev_manifest = manifest_dir/'colab_v2_train_val.json'
test_manifest = manifest_dir/'colab_v2_sealed_test.json'
dev_manifest.write_text(json.dumps(development, indent=2))
test_manifest.write_text(json.dumps(sealed_test, indent=2))
print({'train': TRAIN_FILE_COUNT, 'validation': VALIDATION_FILE_COUNT, 'sealed_test': TEST_FILE_COUNT})

## 3. Securely download and preprocess development data

In [ ]:
import getpass, gc
from smart_scan.data.download import download_manifest
from smart_scan.data.preprocess import preprocess_manifest
from smart_scan.config import load_config, preprocess_kwargs
raw_dir = Path('data/raw')
processed_dir = Path('data/processed_v2_20bands_500us')
token = getpass.getpass('Hugging Face read token (hidden): ')
try:
    receipt = download_manifest(dev_manifest, raw_dir, token=token)
finally:
    del token; gc.collect()
configuration = load_config(Path('configs/v2_20bands_500us.yaml'))
outputs = preprocess_manifest(dev_manifest, raw_dir, processed_dir, **preprocess_kwargs(configuration))
print('Processed', len(outputs), 'development scenarios')

## 4. Train all three candidates and select on validation only

In [ ]:
import yaml
from smart_scan.config import receiver_config, reward_config
from smart_scan.linucb_pipeline import manifest_episode_paths, train_validate_select
search = yaml.safe_load(Path('configs/v2_linucb_search.yaml').read_text())
train_paths = manifest_episode_paths(dev_manifest, processed_dir, 'train')
validation_paths = manifest_episode_paths(dev_manifest, processed_dir, 'val')
selection = train_validate_select(
    train_paths, validation_paths, search['candidates'],
    output_dir=Path('outputs/v2_20bands_500us'), configuration=configuration,
    receiver=receiver_config(configuration), reward=reward_config(configuration),
    jobs=PARALLEL_CANDIDATES, progress=print, resume=True)
print(json.dumps(selection['selected'], indent=2))
print(json.dumps(selection['selection_diagnostics'], indent=2))

## 5. Inspect validation figures of merit and selected-band trace

In [ ]:
import pandas as pd
fom_path = Path('outputs/v2_20bands_500us/selected_validation/figures_of_merit.json')
fom = json.loads(fom_path.read_text())
display(pd.DataFrame(fom['figures_of_merit']))
trace_path = next(Path('outputs/v2_20bands_500us/selected_validation/traces').glob('*_linucb.csv'))
trace = pd.read_csv(trace_path)
display(trace[['time_start_s','band_index','frequency_low_mhz','frequency_high_mhz','detected_pulses','reward']].head(30))

## 6. Optional one-time sealed test

Run only after accepting the frozen validation decision. Do not tune after viewing these results.

In [ ]:
if RUN_FINAL_TEST:
    token = getpass.getpass('Hugging Face token for sealed test (hidden): ')
    try:
        download_manifest(test_manifest, raw_dir, token=token)
    finally:
        del token; gc.collect()
    preprocess_manifest(test_manifest, raw_dir, processed_dir, **preprocess_kwargs(configuration))
    frozen_config = load_config(Path('outputs/v2_20bands_500us/frozen_config.yaml'))
    from smart_scan.linucb_pipeline import evaluate_frozen_linucb
    test_paths = manifest_episode_paths(test_manifest, processed_dir, 'test')
    final = evaluate_frozen_linucb(
        Path('outputs/v2_20bands_500us/frozen_linucb.npz'), test_paths,
        output_dir=Path('outputs/v2_20bands_500us/final_test'),
        receiver=receiver_config(frozen_config), reward=reward_config(frozen_config),
        bootstrap_samples=10000, progress=print, band_log_interval=1000)
    display(pd.DataFrame(final['summary']))
else:
    print('Sealed test was not accessed.')

## Interpretation

Pd, Pfa, and sensitivity are simulator quantities because the source contains PDWs rather than raw IQ/noise. Emitter labels are retained only for offline delay/coverage truth and never enter LinUCB. A 500 μs result is real-time-capable only when measured p99 scheduler latency stays below the dwell deadline.